In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print("Installing bcftools...")
!sudo apt-get update -q
!sudo apt-get install bcftools -y -q

In [ ]:
import os
import subprocess
from tqdm import tqdm
from cyvcf2 import VCF



In [ ]:
print("🔧 Indexing all SG10K chromosomes (1-22)...")
print(f"Input directory: {INPUT_DIR}")
print("=" * 60)

failed_chrs = []

for chr_num in tqdm(range(1, 23), desc="Indexing chromosomes"):
    input_vcf = os.path.join(INPUT_DIR, f"chr{chr_num}.vcf.gz")
    index_file = input_vcf + ".tbi"

    # Check if file exists
    if not os.path.exists(input_vcf):
        print(f"⚠️  Chr{chr_num}: VCF file not found, skipping...")
        failed_chrs.append(chr_num)
        continue

    # Check if already indexed
    if os.path.exists(index_file):
        print(f"✅ Chr{chr_num}: Already indexed, skipping...")
        continue

    # Index the file
    print(f"🔄 Chr{chr_num}: Indexing...")
    exit_code = os.system(f'bcftools index -t "{input_vcf}" 2>/dev/null')

    if exit_code == 0:
        index_size = os.path.getsize(index_file) / (1024**2)
        print(f"✅ Chr{chr_num}: Indexed successfully ({index_size:.2f} MB)")
    else:
        print(f"❌ Chr{chr_num}: Indexing failed!")
        failed_chrs.append(chr_num)

print("\n" + "=" * 60)
print("📊 INDEXING SUMMARY:")
print(f"Total chromosomes: 22")
print(f"Successfully indexed: {22 - len(failed_chrs)}")
if failed_chrs:
    print(f"Failed chromosomes: {failed_chrs}")
else:
    print("✅ All chromosomes indexed successfully!")

In [ ]:
# # ONE TIME FIX, DONOT DO THIS AGAIN
# print("🔧 Fixing line endings in alias file...")
# subprocess.run(f"sed -i 's/\r$//' '{ALIAS_FILE}'", shell=True, check=True)
# print("✅ Alias file converted to Unix format.")

In [ ]:
# --- CONFIGURATION ---
chr_num = "1"
# Adjust filename pattern to match yours (e.g., "chr1.vcf.gz" or "sg10k_chr1.vcf.gz")
input_vcf = os.path.join(INPUT_DIR, f"chr{chr_num}.vcf.gz")
output_vcf = os.path.join(OUTPUT_DIR, f"chr{chr_num}_sas_filtered.vcf.gz")

print(f"Processing Chromosome {chr_num}...")
print(f"Input: {input_vcf}")
print(f"Output: {output_vcf}")

# --- VERIFICATION: BEFORE RUNNING ---
print("📊 Pre-processing verification:")
print(f"Total samples in original: {os.popen(f'bcftools query -l {input_vcf} | wc -l').read().strip()}")
print(f"Indian samples requested: {os.popen(f'wc -l < {ALIAS_FILE}').read().strip()}")
print()

# --- THE COMMAND ---
# 1. -S: Keep only Indian samples
# 2. --force-samples: Ignore if ID missing (safety)
# 3. +fill-tags: Recalculate AF, AC, AN for the new subset

cmd = f"""
bcftools view -S "{ALIAS_FILE}" --force-samples -Ou "{input_vcf}" | \
bcftools plugin fill-tags -Oz -o "{output_vcf}" -- -t AF,AC,AN
"""

# Run it
print("⏳ Running bcftools...")
exit_code = os.system(cmd)
print()

if exit_code == 0:
    print(f"✅ Success! Created: {output_vcf}")

    # Index the new file immediately
    print("Indexing...")
    os.system(f"bcftools index -t '{output_vcf}'")

    # Check size
    size_mb = os.path.getsize(output_vcf) / (1024 * 1024)
    print(f"File Size: {size_mb:.2f} MB")

    # --- VERIFICATION: AFTER RUNNING ---
    print()
    print("📊 Post-processing verification:")
    n_samples = os.popen(f"bcftools query -l '{output_vcf}' | wc -l").read().strip()
    print(f"Final sample count: {n_samples}")

else:
    print("❌ Error processing file. Check paths and logs.")

In [ ]:
# # --- CONFIGURATION ---
# filtered_vcf = os.path.join(OUTPUT_DIR, f"chr1_sas_filtered.vcf.gz")

# print("=" * 80)
# print("VERIFICATION OF FILTERED SG10K CHR1 (Indian samples only)")
# print("=" * 80)

# # ==========================================
# # 1. BASIC FILE CHECKS
# # ==========================================
# print("\n📁 1. FILE INFORMATION:")
# print("-" * 80)
# if os.path.exists(filtered_vcf):
#     size_mb = os.path.getsize(filtered_vcf) / (1024 * 1024)
#     print(f"✅ File exists: {filtered_vcf}")
#     print(f"   File size: {size_mb:.2f} MB")

#     # Check index
#     if os.path.exists(filtered_vcf + ".tbi"):
#         index_size = os.path.getsize(filtered_vcf + ".tbi") / (1024 * 1024)
#         print(f"✅ Index exists: {index_size:.2f} MB")
#     else:
#         print("❌ Index missing!")
# else:
#     print(f"❌ File not found: {filtered_vcf}")

# # ==========================================
# # 2. SAMPLE COUNT VERIFICATION
# # ==========================================
# print("\n📊 2. SAMPLE COUNT VERIFICATION:")
# print("-" * 80)

# # Count samples in filtered VCF
# n_filtered = int(os.popen(f"bcftools query -l '{filtered_vcf}' | wc -l").read().strip()) + 1
# # Count samples in alias file
# n_expected = int(os.popen(f"wc -l < '{ALIAS_FILE}'").read().strip())

# print(f"Expected (from alias file): {n_expected}")
# print(f"Actual (in filtered VCF):   {n_filtered}")

# if n_filtered == n_expected:
#     print(f"✅ MATCH! All {n_filtered} Indian samples present")
# else:
#     print(f"⚠️  MISMATCH! Expected {n_expected} but got {n_filtered}")

# # Show first 10 sample IDs
# print("\nFirst 10 sample IDs in filtered VCF:")
# !bcftools query -l "{filtered_vcf}" | head -10

# # ==========================================
# # 3. VARIANT COUNT COMPARISON
# # ==========================================
# print("\n🧬 3. VARIANT COUNT:")
# print("-" * 80)

# original_vcf = os.path.join(INPUT_DIR, "chr1.vcf.gz")
# n_variants_orig = int(os.popen(f"bcftools view -H '{original_vcf}' | wc -l").read().strip())
# n_variants_filt = int(os.popen(f"bcftools view -H '{filtered_vcf}' | wc -l").read().strip())

# print(f"Original VCF variants: {n_variants_orig:,}")
# print(f"Filtered VCF variants: {n_variants_filt:,}")
# print(f"Difference: {n_variants_orig - n_variants_filt:,} variants removed")
# print(f"Retention rate: {(n_variants_filt/n_variants_orig)*100:.2f}%")

# # ==========================================
# # 4. INFO FIELD VERIFICATION (AF, AC, AN)
# # ==========================================
# print("\n📈 4. INFO FIELD VERIFICATION (AF, AC, AN):")
# print("-" * 80)

# # Check if AF, AC, AN are present and calculated correctly
# print("Checking first 10 variants with AF, AC, AN values:\n")
# !bcftools query -f '%CHROM:%POS\t%REF\t%ALT\tAC=%AC\tAN=%AN\tAF=%AF\n' "{filtered_vcf}" | head -10

# # ==========================================
# # 5. ALLELE FREQUENCY SANITY CHECK
# # ==========================================
# print("\n🔢 5. ALLELE FREQUENCY SANITY CHECKS:")
# print("-" * 80)

# # Expected AN = 2 * number of samples (for diploid)
# expected_an = 2 * n_filtered
# print(f"Expected AN (2 × {n_filtered} samples): {expected_an}")

# # Check actual AN values
# print("\nChecking AN values in first 100 variants:")
# an_check = os.popen(f"bcftools query -f '%AN\\n' '{filtered_vcf}' | head -100 | sort | uniq -c").read()
# print(an_check)

# # Check AF range (should be between 0 and 1)
# print("\nChecking AF range (should be 0-1):")
# !bcftools query -f '%AF\n' "{filtered_vcf}" | head -1000 | awk '{{if($1<0 || $1>1) print "⚠️  Invalid AF:", $1; count++}} END {{if(count==0) print "✅ All AF values in valid range (0-1)"}}'

# # ==========================================
# # 6. COMPARE AF: ORIGINAL vs FILTERED
# # ==========================================
# print("\n🔄 6. ALLELE FREQUENCY COMPARISON (Original vs Filtered):")
# print("-" * 80)
# print("Comparing AF values for first 10 variants:\n")

# # Get AF from original
# !bcftools query -f '%CHROM:%POS\t%AF\n' "{original_vcf}" | head -10 > /tmp/original_af.txt

# # Get AF from filtered
# !bcftools query -f '%CHROM:%POS\t%AF\n' "{filtered_vcf}" | head -10 > /tmp/filtered_af.txt

# # Display comparison
# print("Position\t\tOriginal_AF\tFiltered_AF")
# !paste /tmp/original_af.txt /tmp/filtered_af.txt | awk '{{print $1"\t"$2"\t\t"$4}}'

# print("\n💡 Note: AF values SHOULD be different (Indian-only vs all populations)")

# # ==========================================
# # 7. GENOTYPE CHECK
# # ==========================================
# print("\n🧬 7. GENOTYPE DATA CHECK:")
# print("-" * 80)
# print("First variant with genotypes for first 5 samples:\n")
# !bcftools query -f '%CHROM:%POS\t%REF\t%ALT[\t%GT]\n' "{filtered_vcf}" | head -1 | cut -f1-8

# # ==========================================
# # 8. FINAL SUMMARY
# # ==========================================
# print("\n" + "=" * 80)
# print("✅ VERIFICATION SUMMARY")
# print("=" * 80)
# print(f"Sample count: {n_filtered}/{n_expected} {'✅' if n_filtered == n_expected else '❌'}")
# print(f"Variant count: {n_variants_filt:,}")
# print(f"File size: {size_mb:.2f} MB")
# print(f"Index: {'✅' if os.path.exists(filtered_vcf + '.tbi') else '❌'}")
# print("\nIf all checks pass, you can proceed with batch processing chr2-22!")
# print("=" * 80)

In [ ]:
!sudo apt-get update -q
!sudo apt-get install bcftools -y -q
!pip install cyvcf2 --quiet

import os
import subprocess
from cyvcf2 import VCF

INPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered'
OUTPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized'
REFERENCE_FASTA = '/content/drive/MyDrive/FYP_DATA/DATA/raw_data/reference_fasta/hg38_dna.fa'

# ===================CHANGE THE CHROMOSOME NUMBER ON EVERY RUN ==========================
chr_num = "2"
# =======================================================================================

input_vcf = os.path.join(INPUT_DIR, f'chr{chr_num}_sas_filtered.vcf.gz')

output_vcf = os.path.join(OUTPUT_DIR, f'chr{chr_num}_sas_normalized.vcf.gz')

temp_chr_renamed = os.path.join(OUTPUT_DIR, f'temp_chr{chr_num}_renamed.vcf.gz')
print("=" * 80)
print(f"SG10K NORMALIZATION PIPELINE - CHR{chr_num}")
print("=" * 80)
print(f"Input:  {input_vcf}")
print(f"Output: {output_vcf}")
print(f"Reference: {REFERENCE_FASTA}")

# ============================================================
# STEP 1: Add "chr" prefix to chromosome names
# ============================================================
print("\n" + "=" * 80)
print("STEP 1/4: Adding 'chr' prefix to chromosome names...")
print("=" * 80)

# Create chromosome rename file (only chr1-22 for SG10K)
chr_rename_file = '/tmp/chr_rename_sg10k.txt'
with open(chr_rename_file, 'w') as f:
    for i in range(1, 23):
        f.write(f"{i} chr{i}\n")

print(f"📝 Renaming: {chr_num} → chr{chr_num}")

rename_cmd = f"""
bcftools annotate \
    --rename-chrs {chr_rename_file} \
    -O z \
    -o {temp_chr_renamed} \
    {input_vcf}
"""

print("⏱️  Renaming chromosomes...")
result = subprocess.run(
    rename_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("\n❌ CHROMOSOME RENAMING FAILED!")
    print(result.stderr)
    raise Exception("Chromosome renaming failed")

print("✓ Chromosomes renamed")

# ============================================================
# STEP 2: Index renamed VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 2/4: Indexing renamed VCF...")
print("=" * 80)

print("   Creating index for renamed file...")
index_result = subprocess.run(
    f"bcftools index -t -f {temp_chr_renamed}",
    shell=True,
    capture_output=True,
    text=True
)

if index_result.returncode != 0:
    print(f"   ❌ Indexing failed!")
    print(f"   Error: {index_result.stderr}")
    raise Exception("Failed to create index")
else:
    print(f"   ✓ Index created")

# Verify index exists
if os.path.exists(f"{temp_chr_renamed}.tbi"):
    index_size = os.path.getsize(f"{temp_chr_renamed}.tbi") / 1024
    print(f"   ✓ Index file verified ({index_size:.1f} KB)")
else:
    print("   ❌ Index file not found!")
    raise Exception("Failed to create index")


# ============================================================
# STEP 3: Normalize VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 3/4: Normalizing VCF...")
print("=" * 80)
print("📋 Operations:")
print("   - Left-align indels")
print("   - Split multi-allelic variants (-m-)")
print("   - Check reference alleles")

norm_cmd = f"""
bcftools norm \
    -m- \
    -f {REFERENCE_FASTA} \
    --check-ref w \
    -O z \
    -o {output_vcf} \
    {temp_chr_renamed}
"""

print("\n⏱️  Normalizing (this may take several minutes)...")
result = subprocess.run(
    norm_cmd,
    shell=True,
    capture_output=True,
    text=True
)

# ============================================================
# STEP 4: Index normalized VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 4/4: Indexing normalized VCF...")
print("=" * 80)

index_cmd = f"bcftools index -t -f {output_vcf}"

result = subprocess.run(
    index_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"✓ Index created: {output_vcf}.tbi")
else:
    print("❌ Indexing failed!")
    print(result.stderr)
    raise Exception("Indexing failed")


# ============================================================
# STEP 5: Cleanup temporary files
# ============================================================
print("\n" + "=" * 80)
print("CLEANUP: Removing temporary files...")
print("=" * 80)

try:
    if os.path.exists(temp_chr_renamed):
        os.remove(temp_chr_renamed)
        print(f"✓ Removed: {temp_chr_renamed}")
    if os.path.exists(f"{temp_chr_renamed}.tbi"):
        os.remove(f"{temp_chr_renamed}.tbi")
        print(f"✓ Removed: {temp_chr_renamed}.tbi")
except Exception as e:
    print(f"⚠️  Cleanup warning: {e}")

print("\n" + "=" * 80)
print("🎉 NORMALIZATION COMPLETE!")
print("=" * 80)
print(f"Final output: {output_vcf}")
print(f"Index file: {output_vcf}.tbi")
print("=" * 80)
